In [ ]:
# LE VARIABILI GLOBALI CHE DEVONO ESSERE USATE:
# NOME DELLE CARTELLE DA USARE NELL'ORGANIZZAZIONE DEL DATASET:
A_FOLDER= 'Longitudinal_Studies'
S_FOLDER= 'Super_Resolution'
P_FOLDER = 'Mosaicing'
TEST_FOLDER= 'Test'
REFERENCE_FOLDER= 'Reference'

# Download the data
url = https://projects.ics.forth.gr/cvrl/fire/FIRE.7z
path_to_zip_file = ''

In [1]:
import os
from pathlib import Path
import py7zip
import requests
from rich.console import Console
from typing import Iterator
from contextlib import contextmanager

url_fire_dataset = 'https://projects.ics.forth.gr/cvrl/fire/FIRE.7z'

#zip_request = requests.get(url_fire_dataset)

@contextmanager
def Loading(
    message: str,
) -> Iterator[None]:
    """
    Implement loading animation.

    Parameters
    ----------
    message : str
        The text to print during the animation.

    Returns
    -------
    None

    """
    console = Console()
    try:
        with console.status(f"[bold green]{message}..."):
            yield
    finally:
        console.log(f"[bold green]{message}... Done")


In [28]:
def walk_through_dir(dir_path):
  """
  Walks through dir_path returning its contents.
  Args:
    dir_path (str or pathlib.Path): target directory
  
  Returns:
    A print out of:
      number of subdiretories in dir_path
      number of images (files) in each subdirectory
      name of each subdirectory
  """
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} files in '{dirpath}'.")
 
    

In [44]:
import py7zip
import py7zip.py7zip


fire_path = Path(os.getcwd()).resolve().parent / 'Modules' / 'dataset'
zip_path = fire_path / 'FIRE.7z'
if not fire_path.is_dir():
    os.makedirs(fire_path, exist_ok=True)
else:
    print('the dataser folder already exists')

zip_manager = py7zip.py7zip.Py7zip()

the dataser folder already exists


with Loading('Downloading FIRE dataset'):
    with open(zip_path, 'wb') as f:
        zip_request = requests.get(url_fire_dataset)
        f.write(zip_request.content)
        
destination = fire_path 
source = zip_path
with Loading('Decompressing FIRE dataset'):
    zip_manager.decompress(src = zip_path, dest = fire_path)
    

In [90]:
import os
import py7zr
import requests
import zipfile
from pathlib import Path


def get_dataset(
        dataset_url: str,
        destination_path: Path,
        dataset_name: str = '',
        remove_zip: bool = False
) -> None:
    """
    Download a dataset from a given url and decompress it in a given destination path.

    Parameters
    ----------
    dataset_url : str
        The url of the dataset to download.
    destination_path: Path
        The path where the dataset will be decompressed.
    dataset_name : str, optional
        The name of the dataset file to download.
    remove_zip : bool, optional
        Whether to remove the zip file after decompression.
    """
    if dataset_name == '':
        dataset_name = Path(dataset_url).name

    # Create the destination folder if it does not exist
    if destination_path.is_dir():
        print(f'The {destination_path} folder already exists')
    else:
        os.makedirs(destination_path, exist_ok=True)

    # Control if the dataset_name is present in the folders list of destination_path
    dataset_file = destination_path / dataset_name

    for _, folders, _ in os.walk(destination_path):
        for fold in folders:
            if dataset_name.upper() == fold.upper():
                print(f'[INFO] The {dataset_name} dataset is already extracted in the {destination_path} folder\nNo need to extract it again')
                return None

    # Download and save the dataset from the given url
    if dataset_file.exists():
        print(f'[INFO] {dataset_name} already exists, no need to download it again')
    else:
        zip_request = requests.get(dataset_url)
        with open(dataset_file, 'wb') as f:
            f.write(zip_request.content)

    # Decompress the dataset if not already done
    if dataset_file.suffix == '.7z':
        with py7zr.SevenZipFile(dataset_file, mode='r') as archive:
            archive.extractall(path=destination_path)
    elif dataset_file.suffix == '.zip':
        with zipfile.ZipFile(dataset_file, 'r') as archive:
            archive.extractall(path=destination_path)
    else:
        print(f'[ERROR] Unsupported archive format: {dataset_file.suffix}')
        return None

    if remove_zip:
        os.remove(dataset_file)

    print(f'[INFO] {dataset_name} has been successfully downloaded and extracted to {destination_path}')
    return None

In [157]:
#get_dataset(dataset_url = url_fire_dataset, destination_path = fire_path, dataset_zip = 'FIRE.7z', remove_zip = True)
fire_zip = fire_path / 'FIRE.7z'

get_dataset(dataset_url='https://projects.ics.forth.gr/cvrl/fire/FIRE.7z', 
            destination_path=fire_path,
            remove_zip=False)

The C:\Users\gioco\Desktop\GITHUB_MAIN_FOLDER\ImRegODE\Main_Folder\Modules\dataset folder already exists
[INFO] FIRE.7z already exists, no need to download it again
[INFO] FIRE.7z has been successfully downloaded and extracted to C:\Users\gioco\Desktop\GITHUB_MAIN_FOLDER\ImRegODE\Main_Folder\Modules\dataset


## Categorization Test images - Reference images

In [201]:
import os
import shutil

def organize_folder(parent_folder : Path, # the path folder to organize
                                    categories : list[str], # name of the folders
                                     folder_names: list[str] = []): # the categories of daughter folders
    # aggiungi un file.suffix per il tipo di file da organizzare e spostare nella cartella
    '''given a certain parent dir it reorganize the files respecting the categories:

    assure to give the respective order between the folder name and the categories if given
    EXAMPLE: 
        folder_names = ['truth', 'fake']
        categories ['_1', '_2']
        if _1 corresponds to truth folder
    '''
    if not len(categories):
        print('[ERROR] the categories must be given')
        return 
    
    
    # get the list of files in parent folder
    files = [f for f in parent_folder.glob('*') if f.is_file()]
    if not len(files):
        print(f'[ERROR] there are no files in {parent_folder}')
    # control if categories are indentifiable
    absent_cat =[]
    for cat in categories:
        cat_not_in = False
        for file in files:
            if cat in file.name:
                cat_not_in = True
                break
        if not cat_not_in :
            absent_cat.append(cat)
    for cat in set(absent_cat):
        print(f'[ERROR] the category {cat} is not identifiable in the files')
            
        
    # control if folder names is given:
    if not len(folder_names):
        folder_names = categories
    
    for daughter_folder in folder_names:
        # create the folder if it does not exist
        daughter_folder_path = parent_folder / daughter_folder
        
        if not daughter_folder_path.is_dir():
            os.makedirs(daughter_folder_path, exist_ok=True)
        
        #then move the file if the type of category is in
    path_cat_dict = {k.upper() :v for k , v in zip(categories, folder_names)}

    for file in files:
        cat = file.name[0].upper()
        new_folder = parent_folder / path_cat_dict[cat] / file.name #parent_folder / path_cat_dict[cat]
        shutil.move(src=str(file), dst=str(new_folder))


   
                


In [164]:
image_type = ['Reference','Test']
categories = ['_1', '_2']
fire_images_path = Path.cwd().parent / 'Modules' / 'dataset' / 'FIRE' / 'Images'
organize_folder(parent_folder=fire_images_path, categories=categories, folder_names=image_type)


[ERROR] there are no files in c:\Users\gioco\Desktop\GITHUB_MAIN_FOLDER\ImRegODE\Main_Folder\Modules\dataset\FIRE\Images
[ERROR] the category _1 is not identifiable in the files
[ERROR] the category _2 is not identifiable in the files


In [203]:
image_type = ['Longitudinal_Studies', 'Mosaicing', 'Super_Resolution']
categories = ['A', 'P', 'S']
reference_images_path = Path.cwd().parent / 'Modules' / 'dataset' / 'FIRE' / 'Images' / 'Reference'
organize_folder(parent_folder=reference_images_path, categories=categories, folder_names=image_type)
test_images_path =Path.cwd().parent / 'Modules' / 'dataset' / 'FIRE' / 'Images' / 'Test'
organize_folder(parent_folder=test_images_path, categories=categories, folder_names=image_type)


## Creation of the DATASET

In [215]:
# generate a file path as sample to test all the sequent code:
example_path = Path.cwd().parent / 'Modules' / 'dataset' / 'FIRE' / 'Images' 
list_images = list(example_path.glob('*/*/*.jpg'))
example_image_path = list_images[0]
example_image_path

WindowsPath('c:/Users/gioco/Desktop/GITHUB_MAIN_FOLDER/ImRegODE/Main_Folder/Modules/dataset/FIRE/Images/Reference/Longitudinal_Studies/A01_1.jpg')

In [236]:
def get_dirs_of(file_path : Path,
                 
                )->list[str]:
    '''
    given a certain path it get all the dir as a list of str to get this image
    '''
    if not file_path.is_file():
        image_type, image_deformation = file_path.parts[-2:]
        return [image_type, image_deformation]
        
    image_type, image_deformation, file_name =  file_path.parts[-3:]
    return [image_type, image_deformation, file_name]


In [ ]:
#%pip install -U torch # reinstall using conda by terminal

  Using cached torch-2.6.0-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/204.1 MB 2.8 MB/s eta 0:01:13
   ---------------------------------------- 0.5/204.1 MB 2.8 MB/s eta 0:01:13
   ---------------------------------------- 1.0/204.1 MB 1.5 MB/s eta 0:02:13
   ---------------------------------------- 1.6/204.1 MB 1.9 MB/s eta 0:01:49
   ---------------------------------------- 1.8/204.1 MB 1.8 MB/s eta 0:01:55
   ---------------------------------------- 2.4/204.1 MB 1.9 MB/s eta 0:01:44
    --------------------------------------- 2.6/204.1 MB 1.9 MB/s eta 0:01:46
    --------------------------------------- 2.6/204.1 MB 1.9 MB/s eta 0:01:46
    --------------------------------------- 2.9/204.1 MB 1.7 MB/s eta 0:02:02
    --------------------------------------- 3.1/204.1 MB 1.6 MB/s eta 0:02:09
    -------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imgregode 0.0.0 requires torch==2.4.1, but you have torch 2.6.0 which is incompatible.
torchaudio 2.4.1 requires torch==2.4.1, but you have torch 2.6.0 which is incompatible.
torchvision 0.19.1 requires torch==2.4.1, but you have torch 2.6.0 which is incompatible.


In [249]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
from typing import Tuple, Dict, List
import torch
import random
from pathlib import Path

class ImageRegistrationCustom(Dataset):
    def __init__(self,
                dataset_folder_path: Path, 
                transform=None, 
                target_transform=None)->None:
        self.dataset_path = dataset_folder_path
        path_to_image_types = dataset_folder_path / 'Images' 
        path_to_deformation_types = dataset_folder_path /  'Images' / 'Test' 
        self.reference_images_paths = list(dataset_folder_path.glob('Images/Reference/*/*.jpg'))
        self.test_images_paths = list(dataset_folder_path.glob('Images/Test/*/*.jpg'))
        self.transform = transform
        self.image_types = [d.name for d in path_to_image_types.iterdir() if d.is_dir()]
        self.deformation_types = [d.name  for d in path_to_deformation_types.iterdir() if d.is_dir()]

    # costruisci un dict  per abbreviare long... super res e mosaicing in A, S, P ... ?
    def load_image(self, 
                   index : str | int = 'random',
                   deformation: str|int = '') ->Image.Image: # given an index it get an image
        ''' 
        depending the type there are different possibilities
        depending which type f deformation to visualize
        '''
        if deformation not in self.deformation_types:
            print(f'to see possibile selection use the method .deformation_types')
            raise ValueError(f"Deformation type {deformation} not found in the dataset")
        else:
            if deformation.lower() == 'random':
                images = list(self.dataset_path.glob('*/*/*/*'))
            else:
                images = list(self.dataset_path.glob(f'*/*/*/{deformation}/*'))
        if index == 'random':
            index = random.randint(0, len(images)-1)
        elif isinstance(index, int):
            if index not in range(0, len(images)):
                print('unable to get the desired image, another one casually get from the dirs')
                index = random.randint(0, len(images) - 1)
                images = list(self.dataset_path.glob('*/*/*/*')) 
                image = images[index]
            else:
                image = images[index]
        return Image.open(image)        
        

    def __len__(self,
                info : bool = True):
        '''show the distribution of sample in the dataset path
        then the number of total files in the deepest folder'''
        if info:
            for dirpath, dirnames, filenames in os.walk(self.dataset_path):
                print(f"There are {len(dirnames)} directories and {len(filenames)} files in '{dirpath}'.")
        return len(list(self.dataset_path.glob('*/*/*/*')))
    
    def __getitem__(self,
                    index: bool|int = False,
                    deformation_choice : str = '' 
                    ) -> Tuple[torch.Tensor, torch.Tensor, str]:
        '''get the image pairs from the list of avaiable '''
        if deformation_choice == '' or deformation_choice not in self.deformation_types:
            test_images = list(self.dataset_path.glob('*/Test/*/*'))
            reference_images = list(self.dataset_path.glob('*/Reference/*/*'))
            print('not a valid choice of deformation, use of the entire dataset as batch') 
        else: 
            test_images = list(self.dataset_path.glob(f'*/*/Test/{deformation_choice}/*'))
            reference_images = list(self.dataset_path.glob(f'Images/Reference/{deformation_choice}/*.jpg'))
        if not index or index not in range(0, len(test_images)):
            index = random.randint(0, len(test_images)-1)
            
        test_image = test_images[index]
        reference_image = reference_images[index]
        test_reference,deformation, filename = get_dirs_of(test_image)
        print(f'the selected/chosen image is {filename}, from {deformation} folder of {test_reference}')
        return Tuple[test_image, reference_image, deformation]

NameError: name '_C' is not defined

1

In [246]:
this_path = Path.cwd().parent / 'Modules' / 'dataset' /'FIRE' /'Images'
image_in_this_path = list(this_path.glob('Reference/*/*.jpg'))
#image_in_this_path#, this_path
images = list(this_path.glob('*/*/*'))
images[45]

WindowsPath('c:/Users/gioco/Desktop/GITHUB_MAIN_FOLDER/ImRegODE/Main_Folder/Modules/dataset/FIRE/Images/Reference/Mosaicing/P32_1.jpg')

In [228]:
classes = sorted(entry.name for entry in os.scandir(this_path ) if entry.is_dir())
classes

['Ground Truth', 'Images', 'Masks']

In [234]:
this_path.parts

('c:\\',
 'Users',
 'gioco',
 'Desktop',
 'GITHUB_MAIN_FOLDER',
 'ImRegODE',
 'Main_Folder',
 'Modules',
 'dataset',
 'FIRE',
 'Images')